
# Gold — Clientes ativos por cohort

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_clientes_ativos_cohort`, medindo o percentual de clientes ativos por mês de cadastro.

## Regra de negócio

Cliente ativo = cliente com pelo menos 1 pedido nos últimos 60 dias,
considerando como referência a maior dt_pedido disponível na Silver de pedidos.

Esta métrica mede atividade recente da cohort.

## Fontes

- Silver `ecommerce_clientes`
- Silver `ecommerce_pedidos`

## Cuidados técnicos

- `ecommerce_clientes` é lida como Delta.
- `ecommerce_pedidos` é lida como Delta.
- Pedidos são deduplicados por `id_pedido` antes do cálculo.
- A Gold deve manter uma linha por mês de cohort.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções usadas na criação e validação da Gold.

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    date_sub,
    max as spark_max,
    month,
    round,
    sum as spark_sum,
    to_date,
    to_timestamp,
    trunc,
    when,
    year
)

from pyspark.sql.window import Window

# Define tabelas, caminhos e colunas obrigatórias da Gold.

SILVER_CLIENTES_TABLE = "ecommerce_clientes"
SILVER_PEDIDOS_TABLE = "ecommerce_pedidos"

SILVER_CLIENTES_PATH = f"{SILVER_BASE_PATH}{SILVER_CLIENTES_TABLE}"
SILVER_PEDIDOS_PATH = f"{SILVER_BASE_PATH}{SILVER_PEDIDOS_TABLE}"

GOLD_TABLE = "gold_ecommerce_clientes_ativos_cohort"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

STAGING_TABLE = f"{TARGET_SCHEMA}.stg_{GOLD_TABLE}"
FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

CLIENTES_REQUIRED_COLUMNS = [
    "id_cliente",
    "dt_cadastro"
]

PEDIDOS_REQUIRED_COLUMNS = [
    "id_pedido",
    "id_cliente",
    "silver_processed_at",
    "dt_ultima_atualizacao_status",
    "dt_pedido"
]

GOLD_KEY_COLUMNS = [
    "ano_cadastro",
    "mes_cadastro",
    "data_cohort"
]

JANELA_ATIVIDADE_DIAS = 60

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_CLIENTES_PATH:", SILVER_CLIENTES_PATH)
print("SILVER_PEDIDOS_PATH:", SILVER_PEDIDOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("STAGING_TABLE:", STAGING_TABLE)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers necessárias para montar a Gold.

df_clientes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_CLIENTES_PATH)
)

df_pedidos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PEDIDOS_PATH)
)

total_clientes = df_clientes.count()
total_pedidos = df_pedidos.count()

print("Silver de clientes lida com sucesso.")
print(f"Total clientes: {total_clientes}")

print("Silver de pedidos lida com sucesso em Delta.")
print(f"Total pedidos: {total_pedidos}")

In [0]:
# Valida colunas obrigatórias e chaves principais das fontes.

validate_required_columns(df_clientes, CLIENTES_REQUIRED_COLUMNS)
validate_required_columns(df_pedidos, PEDIDOS_REQUIRED_COLUMNS)

clientes_distintos = (
    df_clientes
    .select(col("id_cliente").cast("int").alias("id_cliente"))
    .distinct()
    .count()
)

pedidos_distintos = (
    df_pedidos
    .select(col("id_pedido").cast("int").alias("id_pedido"))
    .distinct()
    .count()
)

clientes_duplicados = total_clientes - clientes_distintos
pedidos_duplicados = total_pedidos - pedidos_distintos

print(f"Total clientes: {total_clientes}")
print(f"Clientes distintos: {clientes_distintos}")
print(f"Clientes duplicados: {clientes_duplicados}")

print(f"Total pedidos: {total_pedidos}")
print(f"Pedidos distintos: {pedidos_distintos}")
print(f"Pedidos duplicados: {pedidos_duplicados}")

if clientes_duplicados > 0:
    raise Exception("Erro: existem clientes duplicados por id_cliente.")

if pedidos_distintos == 0:
    raise Exception("Erro: não foram encontrados pedidos válidos por id_pedido.")

print("Validação OK: fontes mínimas conferidas.")

In [0]:
# Deduplica pedidos por id_pedido mantendo o registro mais recente.

from pyspark.sql.functions import row_number

df_pedidos_base = (
    df_pedidos
    .withColumn("id_pedido_int", col("id_pedido").cast("int"))
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
    .withColumn("silver_processed_at_ts", to_timestamp(col("silver_processed_at")))
    .withColumn("dt_ultima_atualizacao_status_ts", to_timestamp(col("dt_ultima_atualizacao_status")))
    .withColumn("dt_pedido_ts", to_timestamp(col("dt_pedido")))
)

window_pedidos_dedup = (
    Window
    .partitionBy("id_pedido_int")
    .orderBy(
        col("silver_processed_at_ts").desc_nulls_last(),
        col("dt_ultima_atualizacao_status_ts").desc_nulls_last(),
        col("dt_pedido_ts").desc_nulls_last()
    )
)

df_pedidos_dedup = (
    df_pedidos_base
    .withColumn("rn", row_number().over(window_pedidos_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Pedidos deduplicados por id_pedido.")

In [0]:
# Define clientes ativos com pedido nos últimos 60 dias

from datetime import timedelta

data_referencia_pedidos = (
    df_pedidos_dedup
    .agg(spark_max(to_date(col("dt_pedido_ts"))).alias("data_referencia"))
    .collect()[0]["data_referencia"]
)

data_inicio_janela = data_referencia_pedidos - timedelta(days=JANELA_ATIVIDADE_DIAS)

clientes_com_pedido_recente = (
    df_pedidos_dedup
    .filter(col("id_cliente_int").isNotNull())
    .filter(to_date(col("dt_pedido_ts")) >= data_inicio_janela)
    .select(col("id_cliente_int").alias("id_cliente"))
    .distinct()
    .withColumn("cliente_tem_pedido_recente", when(col("id_cliente").isNotNull(), 1))
)

print(f"Data referência dos pedidos: {data_referencia_pedidos}")
print(f"Início da janela de atividade: {data_inicio_janela}")
print(f"Janela considerada: últimos {JANELA_ATIVIDADE_DIAS} dias")
print(f"Clientes ativos na janela: {clientes_com_pedido_recente.count()}")

In [0]:
# Valida se a deduplicação deixou apenas um registro por pedido.

total_pedidos_dedup = df_pedidos_dedup.count()

pedidos_distintos_dedup = (
    df_pedidos_dedup
    .select("id_pedido_int")
    .distinct()
    .count()
)

pedidos_duplicados_dedup = total_pedidos_dedup - pedidos_distintos_dedup

pedidos_id_nulo = (
    df_pedidos_dedup
    .filter(col("id_pedido_int").isNull())
    .count()
)

clientes_id_nulo_em_pedidos = (
    df_pedidos_dedup
    .filter(col("id_cliente_int").isNull())
    .count()
)

print(f"Total pedidos original: {total_pedidos}")
print(f"Total pedidos após deduplicação: {total_pedidos_dedup}")
print(f"Pedidos distintos após deduplicação: {pedidos_distintos_dedup}")
print(f"Pedidos duplicados restantes: {pedidos_duplicados_dedup}")
print(f"Pedidos com id_pedido nulo: {pedidos_id_nulo}")
print(f"Pedidos com id_cliente nulo: {clientes_id_nulo_em_pedidos}")

if pedidos_duplicados_dedup > 0:
    raise Exception("Erro: ainda existem pedidos duplicados por id_pedido.")

if pedidos_id_nulo > 0:
    raise Exception("Erro: existem pedidos com id_pedido nulo.")

print("Validação OK: pedidos deduplicados corretamente.")

In [0]:
# Cria base de clientes com flag de cliente ativo recente

df_clientes_com_flag_ativo = (
    df_clientes
    .select(
        col("id_cliente").cast("int").alias("id_cliente"),
        col("dt_cadastro")
    )
    .join(
        clientes_com_pedido_recente,
        on="id_cliente",
        how="left"
    )
    .withColumn(
        "cliente_ativo",
        when(col("cliente_tem_pedido_recente").isNotNull(), 1).otherwise(0)
    )
    .withColumn("data_cohort", trunc(to_date(col("dt_cadastro")), "MM"))
    .withColumn("ano_cadastro", year(col("dt_cadastro")))
    .withColumn("mes_cadastro", month(col("dt_cadastro")))
    .drop("cliente_tem_pedido_recente")
)

print("Base de clientes com flag de ativo recente criada.")

In [0]:
# Valida se a base manteve um registro por cliente.

total_clientes_base = df_clientes_com_flag_ativo.count()

clientes_distintos_base = (
    df_clientes_com_flag_ativo
    .select("id_cliente")
    .distinct()
    .count()
)

clientes_duplicados_base = total_clientes_base - clientes_distintos_base

clientes_sem_cohort = (
    df_clientes_com_flag_ativo
    .filter(col("data_cohort").isNull())
    .count()
)

print(f"Total clientes original: {total_clientes}")
print(f"Total clientes na base: {total_clientes_base}")
print(f"Clientes distintos na base: {clientes_distintos_base}")
print(f"Clientes duplicados na base: {clientes_duplicados_base}")
print(f"Clientes sem data_cohort: {clientes_sem_cohort}")

if total_clientes_base != total_clientes:
    raise Exception("Erro: a base alterou a quantidade de clientes.")

if clientes_duplicados_base > 0:
    raise Exception("Erro: existem clientes duplicados na base.")

if clientes_sem_cohort > 0:
    raise Exception("Erro: existem clientes sem data_cohort.")

print("Validação OK: base manteve 1 registro por cliente.")

In [0]:
# Cria a Gold de clientes ativos recentes por mês de cohort.

df_gold = (
    df_clientes_com_flag_ativo
    .groupBy(
        "ano_cadastro",
        "mes_cadastro",
        "data_cohort"
    )
    .agg(
        count("id_cliente").alias("qtd_clientes_cadastrados"),
        spark_sum("cliente_ativo").alias("qtd_clientes_ativos")
    )
    .withColumn(
        "qtd_clientes_inativos",
        col("qtd_clientes_cadastrados") - col("qtd_clientes_ativos")
    )
    .withColumn(
        "percentual_clientes_ativos",
        round(
            (col("qtd_clientes_ativos") / col("qtd_clientes_cadastrados")) * 100,
            2
        )
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("ano_cadastro", "mes_cadastro")
)

print("Gold criada em memória.")
display(df_gold)

In [0]:
# Valida totais, duplicidade de cohort e nulos principais da Gold.

total_linhas_gold = df_gold.count()

total_cohorts_distintos = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

duplicados_cohort = total_linhas_gold - total_cohorts_distintos

validacao_totais_gold = (
    df_gold
    .agg(
        spark_sum("qtd_clientes_cadastrados").alias("total_clientes_cadastrados"),
        spark_sum("qtd_clientes_ativos").alias("total_clientes_ativos"),
        spark_sum("qtd_clientes_inativos").alias("total_clientes_inativos")
    )
    .collect()[0]
)

total_gold_clientes = validacao_totais_gold["total_clientes_cadastrados"]
total_gold_ativos = validacao_totais_gold["total_clientes_ativos"]
total_gold_inativos = validacao_totais_gold["total_clientes_inativos"]

nulos_gold = (
    df_gold
    .filter(
        col("ano_cadastro").isNull() |
        col("mes_cadastro").isNull() |
        col("data_cohort").isNull() |
        col("qtd_clientes_cadastrados").isNull() |
        col("qtd_clientes_ativos").isNull() |
        col("qtd_clientes_inativos").isNull() |
        col("percentual_clientes_ativos").isNull()
    )
    .count()
)

print(f"Total clientes base: {total_clientes_base}")
print(f"Total clientes na Gold: {total_gold_clientes}")
print(f"Total clientes ativos na Gold: {total_gold_ativos}")
print(f"Total clientes inativos na Gold: {total_gold_inativos}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Cohorts duplicados: {duplicados_cohort}")
print(f"Linhas com nulos principais: {nulos_gold}")

if total_gold_clientes != total_clientes_base:
    raise Exception("Erro: total de clientes da Gold não fecha com a base.")

if total_gold_ativos + total_gold_inativos != total_gold_clientes:
    raise Exception("Erro: ativos + inativos não fecha com cadastrados.")

if duplicados_cohort > 0:
    raise Exception("Erro: existem cohorts duplicados na Gold.")

if nulos_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_cohorts_saved = (
    df_gold_saved
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

duplicados_cohort_saved = total_linhas_gold_saved - total_cohorts_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_clientes_cadastrados").alias("total_clientes_cadastrados"),
        spark_sum("qtd_clientes_ativos").alias("total_clientes_ativos"),
        spark_sum("qtd_clientes_inativos").alias("total_clientes_inativos")
    )
    .collect()[0]
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Cohorts duplicados Gold Delta: {duplicados_cohort_saved}")
print(f"Total clientes base: {total_clientes_base}")
print(f"Total clientes Gold Delta: {validacao_gold_saved['total_clientes_cadastrados']}")
print(f"Total ativos Gold Delta: {validacao_gold_saved['total_clientes_ativos']}")
print(f"Total inativos Gold Delta: {validacao_gold_saved['total_clientes_inativos']}")

if validacao_gold_saved["total_clientes_cadastrados"] != total_clientes_base:
    raise Exception("Erro: total de clientes da Gold Delta não confere.")

if (
    validacao_gold_saved["total_clientes_ativos"] +
    validacao_gold_saved["total_clientes_inativos"]
    != validacao_gold_saved["total_clientes_cadastrados"]
):
    raise Exception("Erro: ativos + inativos não fecha com cadastrados na Gold Delta.")

if duplicados_cohort_saved > 0:
    raise Exception("Erro: existem cohorts duplicados na Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("ano_cadastro").cast("int").alias("ano_cadastro"),
        col("mes_cadastro").cast("int").alias("mes_cadastro"),
        col("data_cohort").cast("date").alias("data_cohort"),
        col("qtd_clientes_cadastrados").cast("int").alias("qtd_clientes_cadastrados"),
        col("qtd_clientes_ativos").cast("int").alias("qtd_clientes_ativos"),
        col("qtd_clientes_inativos").cast("int").alias("qtd_clientes_inativos"),
        col("percentual_clientes_ativos").cast("decimal(5,2)").alias("percentual_clientes_ativos"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("ano_cadastro", "mes_cadastro"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_cohorts_final = (
    df_final
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

cohorts_duplicados_final = total_linhas_final - total_cohorts_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_clientes_cadastrados").alias("total_clientes_cadastrados"),
        spark_sum("qtd_clientes_ativos").alias("total_clientes_ativos"),
        spark_sum("qtd_clientes_inativos").alias("total_clientes_inativos")
    )
    .collect()[0]
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Cohorts duplicados tabela final: {cohorts_duplicados_final}")
print(f"Total clientes base: {total_clientes_base}")
print(f"Total clientes tabela final: {validacao_final['total_clientes_cadastrados']}")
print(f"Total ativos tabela final: {validacao_final['total_clientes_ativos']}")
print(f"Total inativos tabela final: {validacao_final['total_clientes_inativos']}")

if validacao_final["total_clientes_cadastrados"] != total_clientes_base:
    raise Exception("Erro: total de clientes da tabela final não confere.")

if (
    validacao_final["total_clientes_ativos"] +
    validacao_final["total_clientes_inativos"]
    != validacao_final["total_clientes_cadastrados"]
):
    raise Exception("Erro: ativos + inativos não fecha com cadastrados na tabela final.")

if cohorts_duplicados_final > 0:
    raise Exception("Erro: existem cohorts duplicados na tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")